# Verify simplex / cerebellar coordinates in injection-structured.json

This notebook reads the raw JSON and prints exactly what is stored. No filtering,
no interpretation baked in — you see the actual records and decide.

Set the path in Cell 1, then Run All.

In [2]:
# Cell 1 — load the file
import json, os

PATH = 'C:\\Users\\asathyanesan1\\Documents\\GitHub\\Neuroinjector\\react-app\\public\\data\\injection-structured.json'   # <-- edit if needed

d = json.load(open(PATH, encoding='utf-8'))
print('file:', os.path.abspath(PATH))
print('size (MB):', round(os.path.getsize(PATH)/1024/1024, 2))
print('total papers:', len(d))
# how many targets total
nt = sum(len(r.get('targets', [])) for r in d.values() if isinstance(r, dict))
print('total targets:', nt)

file: C:\Users\asathyanesan1\Documents\GitHub\Neuroinjector\react-app\public\data\injection-structured.json
size (MB): 5.82
total papers: 14056
total targets: 11771


In [3]:
# Cell 2 — dump the exact record for PMID 31934857 (the disputed one)
pmid = '31934857'
rec = d.get(pmid)
if rec is None:
    print(pmid, 'NOT in file')
else:
    print(json.dumps(rec, indent=2, ensure_ascii=False))

{
  "pmid": "31934857",
  "targets": [
    {
      "region_verbatim": "cerebellar Simplex lobule",
      "ccf_region": "Cerebellar Simplex Lobule",
      "ap_mm": -5.8,
      "ml_mm": 2.2,
      "dv_mm": null,
      "reference": "bregma",
      "volume_nl": 0.025,
      "rate_nl_min": 0.025,
      "source_quote": "cerebellar Simplex lobule, AP: −5.8 mm ML: 2.2 mm"
    },
    {
      "region_verbatim": "lobule VI",
      "ccf_region": "Cerebellar Lobule VI",
      "ap_mm": -7.4,
      "ml_mm": 0.0,
      "dv_mm": null,
      "reference": "bregma",
      "volume_nl": 0.025,
      "rate_nl_min": 0.025,
      "source_quote": "lobule VI, AP: −7.4 mm, ML: 0.0 mm"
    },
    {
      "region_verbatim": "cortex",
      "ccf_region": "Cortex",
      "ap_mm": 1.4,
      "ml_mm": 1.5,
      "dv_mm": null,
      "reference": "bregma",
      "volume_nl": 0.025,
      "rate_nl_min": 0.025,
      "source_quote": "cortex, AP: 1.4 mm, ML: 1.5 mm"
    },
    {
      "region_verbatim": "dorsal striatum",


In [4]:
# Cell 3 — EVERY target whose region text mentions simplex / simple lobule
import re
rx = re.compile(r'simplex|simple\s*lobule|lobulus\s*simplex|\bSIM\b', re.I)

hits = []
for k, rec in d.items():
    if not isinstance(rec, dict):
        continue
    for t in rec.get('targets', []):
        if not isinstance(t, dict):
            continue
        blob = f"{t.get('ccf_region','')} {t.get('region_verbatim','')} {t.get('source_quote','')}"
        if rx.search(blob):
            hits.append((k, t))

print(f'{len(hits)} simplex-matching targets across {len(set(h[0] for h in hits))} papers\n')
for pmid, t in hits:
    print('='*80)
    print('PMID:', pmid)
    print('  ccf_region     :', t.get('ccf_region'))
    print('  region_verbatim:', t.get('region_verbatim'))
    print('  AP / ML / DV   :', t.get('ap_mm'), '/', t.get('ml_mm'), '/', t.get('dv_mm'))
    print('  reference      :', t.get('reference'))
    print('  volume_nl      :', t.get('volume_nl'))
    print('  rate_nl_min    :', t.get('rate_nl_min'))
    print('  source_quote   :', t.get('source_quote'))

9 simplex-matching targets across 7 papers

PMID: 30079375
  ccf_region     : dorsal hippocampus
  region_verbatim: dHPC
  AP / ML / DV   : -2.2 / 2.0 / -2.0 to -1.8
  reference      : bregma
  volume_nl      : 300
  rate_nl_min    : 100
  source_quote   : Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm anteroposterior, ±2.0 mm ML, −2.0 and −1.8 mm DV (0.3 μl of purified virus was infused at each DV site at a rate of 0.1 μl/min).
PMID: 30079375
  ccf_region     : dorsal hippocampus
  region_verbatim: dHPC
  AP / ML / DV   : -2.2 / -2.0 / -2.0 to -1.8
  reference      : bregma
  volume_nl      : 300
  rate_nl_min    : 100
  source_quote   : Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm anteroposterior, ±2.0 mm ML, −2.0 and −1.8 mm DV (0.3 μl of purified virus was infused at each DV site at a rate of 0.1 μl/min).
PMID: 31934

In [5]:
# Cell 4 — does each numeric value actually appear in its own source_quote?
# (catches fabricated / cross-contaminated numbers: a value with no textual support)
def in_quote(val, quote):
    if val is None or quote is None:
        return None
    q = str(quote).replace('\u2212', '-')
    # match the number ignoring trailing zeros (2.2 matches '2.2', '2.20')
    s = str(val)
    nums = re.findall(r'-?\d+\.?\d*', s)
    return all(n in q for n in nums) if nums else None

print('For each simplex target: is the value literally present in its source_quote?\n')
for pmid, t in hits:
    q = t.get('source_quote') or ''
    print(f"PMID {pmid} | {t.get('ccf_region')}")
    for field in ('ap_mm', 'ml_mm', 'dv_mm', 'volume_nl', 'rate_nl_min'):
        v = t.get(field)
        chk = in_quote(v, q)
        tag = {True: 'in quote', False: 'NOT IN QUOTE', None: '(null / n-a)'}[chk]
        print(f"    {field:12} = {str(v):20} -> {tag}")
    print('    quote:', q[:160])
    print()

For each simplex target: is the value literally present in its source_quote?

PMID 30079375 | dorsal hippocampus
    ap_mm        = -2.2                 -> in quote
    ml_mm        = 2.0                  -> in quote
    dv_mm        = -2.0 to -1.8         -> in quote
    volume_nl    = 300                  -> NOT IN QUOTE
    rate_nl_min  = 100                  -> NOT IN QUOTE
    quote: Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm anteroposterior, ±2.0 mm ML, −

PMID 30079375 | dorsal hippocampus
    ap_mm        = -2.2                 -> in quote
    ml_mm        = -2.0                 -> in quote
    dv_mm        = -2.0 to -1.8         -> in quote
    volume_nl    = 300                  -> NOT IN QUOTE
    rate_nl_min  = 100                  -> NOT IN QUOTE
    quote: Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm an

In [6]:
# Cell 5 — implausibly small volume / rate across the WHOLE file (the 0.025 class)
small = []
for k, rec in d.items():
    if not isinstance(rec, dict):
        continue
    for t in rec.get('targets', []):
        if not isinstance(t, dict):
            continue
        v, r = t.get('volume_nl'), t.get('rate_nl_min')
        if (isinstance(v, (int, float)) and 0 < v < 0.1) or (isinstance(r, (int, float)) and 0 < r < 0.1):
            small.append((k, t.get('ccf_region'), v, r, (t.get('source_quote') or '')[:100]))

print(f'targets with volume<0.1 nL OR rate<0.1 nL/min : {len(small)}\n')
for k, reg, v, r, q in small[:40]:
    print(f'  {k} | {reg} | vol={v} rate={r}')
    print(f'       quote: {q}')

targets with volume<0.1 nL OR rate<0.1 nL/min : 1199

  19327389 | Parabrachial nucleus | vol=0.06 rate=None
       quote: A glass micropipette (20–25 μm tip diameter) filled with 5% Fluorogold (FG) was lowered into the PbN
  19380255 | Perirhinal cortex | vol=0.05 rate=None
       quote: Average ± SEM stereotaxic coordinates of the microinjection sites were 4.92 ± 0.05 mm posterior, 0.7
  20053381 | subventricular zone | vol=0.05 rate=None
       quote: stereotaxic coordinates 0.7–0.9 mm lateral to bregma, 1.1–1.3 mm rostral to bregma and 2.4–2.6 mm de
  20371824 | Insular cortex | vol=0.5 rate=0.014285714285714285
       quote: two stainless-steel guide cannulae (23 gauge; Small Parts, Inc, Miami Lakes, FL) were implanted bila
  20371824 | Insular cortex | vol=0.5 rate=0.014285714285714285
       quote: two stainless-steel guide cannulae (23 gauge; Small Parts, Inc, Miami Lakes, FL) were implanted bila
  20371824 | Hippocampus | vol=0.5 rate=0.014285714285714285
       quote: two sta

In [7]:
# Cell 6 — sanity distribution of all non-null volumes & rates (spot the outliers)
vols  = [t.get('volume_nl')  for r in d.values() if isinstance(r, dict)
         for t in r.get('targets', []) if isinstance(t, dict)
         and isinstance(t.get('volume_nl'), (int, float))]
rates = [t.get('rate_nl_min') for r in d.values() if isinstance(r, dict)
         for t in r.get('targets', []) if isinstance(t, dict)
         and isinstance(t.get('rate_nl_min'), (int, float))]

def summarize(name, xs):
    xs = sorted(xs)
    if not xs:
        print(name, ': none'); return
    n = len(xs)
    print(f'{name}: n={n}  min={xs[0]}  p05={xs[int(n*0.05)]}  median={xs[n//2]}  p95={xs[int(n*0.95)]}  max={xs[-1]}')
    print(f'    below 0.1 : {sum(1 for x in xs if x < 0.1)}')
    print(f'    0.1 - 1   : {sum(1 for x in xs if 0.1 <= x < 1)}')
    print(f'    1 - 50    : {sum(1 for x in xs if 1 <= x < 50)}')
    print(f'    >= 50     : {sum(1 for x in xs if x >= 50)}')

summarize('volume_nl ', vols)
print()
summarize('rate_nl_min', rates)

volume_nl : n=8451  min=0.001  p05=0.1  median=75  p95=2500  max=100000000
    below 0.1 : 395
    0.1 - 1   : 2846
    1 - 50    : 822
    >= 50     : 4388

rate_nl_min: n=6278  min=0.001  p05=0.03  median=50  p95=500  max=1000000
    below 0.1 : 911
    0.1 - 1   : 1067
    1 - 50    : 1110
    >= 50     : 3190


In [9]:
import json, re
d = json.load(open('C:\\Users\\asathyanesan1\\Documents\\GitHub\\Neuroinjector\\react-app\\public\\data\\injection-structured.json'))

rx_simplex = re.compile(r'simplex|simple\s*lobule|lobulus\s*simplex|\bHVI\b|\bSIM\b', re.I)
rx_cereb   = re.compile(r'cerebell|purkinje|lobule|vermis|crus|folium|HVI', re.I)
rx_hsv     = re.compile(r'herpes|\bHSV\b|simplex\s*virus', re.I)

print("Papers matching 'simplex' — classified:\n")
for pmid, rec in d.items():
    for t in rec.get('targets', []):
        if not isinstance(t, dict): continue
        blob = f"{t.get('ccf_region','')} {t.get('region_verbatim','')} {t.get('source_quote','')}"
        if not rx_simplex.search(blob): continue
        is_hsv   = bool(rx_hsv.search(blob))
        is_cereb = bool(rx_cereb.search(blob))
        # verdict
        if is_hsv and not is_cereb:
            verdict = "REJECT: herpes simplex virus (not the lobule)"
        elif not is_cereb:
            verdict = "REJECT: no cerebellar context (landmark/other)"
        else:
            verdict = "KEEP: real cerebellar simplex"
        print(f"{pmid} | {t.get('ccf_region')}")
        print(f"    -> {verdict}")
        print(f"    quote: {t.get('source_quote','')[:130]}\n")

Papers matching 'simplex' — classified:

30079375 | dorsal hippocampus
    -> REJECT: herpes simplex virus (not the lobule)
    quote: Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm 

30079375 | dorsal hippocampus
    -> REJECT: herpes simplex virus (not the lobule)
    quote: Stereotaxic surgery was performed to inject herpes simplex virus (HSV) vectors into bilateral dHPC, as follows: 7° angle; −2.2 mm 

31934857 | Cerebellar Simplex Lobule
    -> KEEP: real cerebellar simplex
    quote: cerebellar Simplex lobule, AP: −5.8 mm ML: 2.2 mm

34250904 | ventral cochlear nucleus
    -> KEEP: real cerebellar simplex
    quote: the VCN was located by stereotactic coordinates (0.7 mm lateral, 0.95 mm rostral, and 4.0 mm depth) starting from the surface junc

37248339 | Cerebellar cortex
    -> KEEP: real cerebellar simplex
    quote: optical fibers ... implanted into the primary fissure, between Lob 4/5 and Sim